# Task 3 - LSTM (Kaggle/Colab)
Train LSTM for the three required areas and save predictions + metrics.

In [ ]:
# Setup and data load
from pathlib import Path
import time
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow import keras

DATA_PATH = Path("/kaggle/input/telecom-traffic/city_traffic.parquet")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Parquet not found at {DATA_PATH}")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load the parquet once for the three target series.
df = pd.read_parquet(DATA_PATH)
rename_map = {
    "squareid": "square_id",
    "square": "square_id",
    "time": "time_interval",
    "timestamp": "time_interval",
    "internet": "internet_traffic",
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})
required_cols = ["square_id", "time_interval", "internet_traffic"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise KeyError(f"Missing columns: {missing}. Available: {df.columns.tolist()}")

df["time_interval"] = pd.to_datetime(df["time_interval"], utc=True)
df = df.sort_values(["square_id", "time_interval"]).reset_index(drop=True)


def get_area_series(data: pd.DataFrame, square_id: int) -> pd.Series:
    s = data[data["square_id"] == square_id].sort_values("time_interval")
    series = pd.to_numeric(s.set_index("time_interval")["internet_traffic"], errors="coerce").astype(float)
    series = series.replace([np.inf, -np.inf], np.nan)

    # Resample to a fixed 10-minute grid so scaling and sequence creation stay stable.
    series = series.resample("10min").mean()
    series = series.interpolate(limit_direction="both")
    series = series.ffill().bfill().dropna()
    return series


def split_train_test(series: pd.Series):
    train_end = pd.Timestamp("2013-12-15 23:50:00", tz="UTC")
    test_start = pd.Timestamp("2013-12-16 00:00:00", tz="UTC")
    test_end = pd.Timestamp("2013-12-22 23:50:00", tz="UTC")
    train = series.loc[:train_end]
    test = series.loc[test_start:test_end]
    return train, test


def make_sequences(values: np.ndarray, seq_len: int):
    x, y = [], []
    for i in range(len(values) - seq_len):
        x.append(values[i : i + seq_len])
        y.append(values[i + seq_len])
    return np.array(x), np.array(y)


target_squares = [int(df.groupby("square_id")["internet_traffic"].sum().idxmax()), 4159, 4556]
SEQ_LEN = 144

In [ ]:
# LSTM training + evaluation
def evaluate_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    y_true = np.asarray(y_true, dtype=float).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=float).reshape(-1)
    n = min(len(y_true), len(y_pred))
    y_true = y_true[:n]
    y_pred = y_pred[:n]
    mae = float(np.mean(np.abs(y_true - y_pred)))
    mape = float(np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8))) * 100.0)
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    return {"MAE": mae, "MAPE": mape, "RMSE": rmse}

lstm_results = []
for square_id in target_squares:
    series = get_area_series(df, int(square_id))
    if series.empty or not np.isfinite(series.to_numpy()).all():
        print(f"Skipping square {square_id}: unusable series after cleaning")
        continue

    train, test = split_train_test(series)

    if len(train) <= SEQ_LEN + 1 or len(test) <= SEQ_LEN:
        print(f"Skipping square {square_id}: insufficient points after preprocessing")
        continue

    scaler = MinMaxScaler()
    train_values = train.values.reshape(-1, 1)
    test_values = test.values.reshape(-1, 1)
    if not np.isfinite(train_values).all() or not np.isfinite(test_values).all():
        print(f"Skipping square {square_id}: non-finite values remain after cleaning")
        continue

    train_scaled = scaler.fit_transform(train_values).astype(np.float32).squeeze()
    test_scaled = scaler.transform(test_values).astype(np.float32).squeeze()

    if not np.isfinite(train_scaled).all() or not np.isfinite(test_scaled).all():
        print(f"Skipping square {square_id}: scaling produced non-finite values")
        continue

    # Build training windows only from the training history.
    x_train, y_train = make_sequences(train_scaled, SEQ_LEN)

    # Build test windows from the last train window plus the test horizon.
    test_context = np.concatenate([train_scaled[-SEQ_LEN:], test_scaled])
    x_test, y_test = make_sequences(test_context, SEQ_LEN)

    x_train = x_train[..., np.newaxis]
    x_test = x_test[..., np.newaxis]

    # Fail fast if any NaN/Inf made it into the tensors.
    if not np.isfinite(x_train).all() or not np.isfinite(y_train).all():
        print(f"Skipping square {square_id}: non-finite values detected in training tensors")
        continue
    if not np.isfinite(x_test).all():
        print(f"Skipping square {square_id}: non-finite values detected in test tensors")
        continue

    model = keras.Sequential([
        keras.layers.Input(shape=(SEQ_LEN, 1)),
        keras.layers.LSTM(64, return_sequences=False),
        keras.layers.Dense(1),
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse")

    start = time.perf_counter()
    model.fit(x_train, y_train, epochs=10, batch_size=128, verbose=1)
    train_time = time.perf_counter() - start

    start = time.perf_counter()
    pred_scaled = model.predict(x_test, verbose=0).squeeze()
    infer_time = time.perf_counter() - start

    # Predict exactly the same horizon used to build x_test.
    y_pred = scaler.inverse_transform(np.asarray(pred_scaled).reshape(-1, 1)).squeeze()
    y_true = test.values[: len(y_pred)]
    metrics = evaluate_metrics(y_true, y_pred)
    metrics.update({"square_id": int(square_id), "train_seconds": train_time, "infer_seconds": infer_time})
    lstm_results.append(metrics)

    out = pd.DataFrame({
        "time_interval": test.index[: len(y_pred)],
        "y_true": y_true,
        "y_pred": y_pred,
        "model": "LSTM",
        "square_id": int(square_id),
    })
    out.to_csv(OUTPUT_DIR / f"lstm_{int(square_id)}.csv", index=False)

metrics_df = pd.DataFrame(lstm_results)
metrics_df.to_csv(OUTPUT_DIR / "metrics_lstm.csv", index=False)
metrics_df